# Hierarchical LLM Semantic Graph Builder — Interactive Notebook

Двухпроходный пайплайн:
1. **Pass 1** — иерархический обзор корпуса (chunk summaries → агрегация → root).
2. **Pass 2** — extraction триплетов с глобальным контекстом из дерева, затем ER → importance-фильтр → multi_clustered_graph.

Каждая ячейка — отдельная стадия. Можно запускать последовательно и инспектировать промежуточные результаты.

## Setup

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch

In [2]:
import sys, os, logging
from pathlib import Path

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

_root = Path().resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
_here = Path().resolve()
if str(_here) in sys.path:
    sys.path.remove(str(_here))

# os.environ['YANDEX_CLOUD_API_KEY'] = '<your-key>'

from hierarchical_llm_version.config_schema import load_config
config = load_config('config.yaml')
print(config.model_dump_json(indent=2))

{
  "llm": {
    "api_key_env": "YANDEX_CLOUD_API_KEY",
    "api_key": "",
    "base_url": "https://ai.api.cloud.yandex.net/v1",
    "folder": "b1gpiug3vgbpe1cb4e5c",
    "cheap": {
      "model_id": "deepseek-v32/latest",
      "temperature": 0.3,
      "max_output_tokens": 1500,
      "instructions": ""
    },
    "strong": {
      "model_id": "deepseek-v32/latest",
      "temperature": 0.3,
      "max_output_tokens": 3000,
      "instructions": ""
    },
    "extraction": {
      "model_id": "deepseek-v32/latest",
      "temperature": 0.3,
      "max_output_tokens": 4000,
      "instructions": ""
    },
    "max_concurrency": 4,
    "max_retries": 3,
    "retry_base_delay": 1.0
  },
  "embedding": {
    "model_name": "intfloat/multilingual-e5-large",
    "device": "cpu",
    "prefix": null
  },
  "preprocessing": {
    "language": "ru",
    "chunk_size": 5,
    "overlap_size": 1
  },
  "pass1": {
    "summary_prompt": "prompts/chunk_summary_ru.txt",
    "aggregation_prompt": "prompt

In [ ]:
config.llm.api_key = ""

In [4]:
from hierarchical_llm_version.models.llm_client import LLMClient
from hierarchical_llm_version.models.embedder import Embedder

llm = LLMClient(config.llm)
print(f'LLM: cheap={config.llm.cheap.model_id}, strong={config.llm.strong.model_id}, extraction={config.llm.extraction.model_id}, concurrency={config.llm.max_concurrency}')

embedder = Embedder(config.embedding)
print(f'Embedder: {config.embedding.model_name} (dim={embedder.dim})')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-11 19:58:35,178 [INFO] datasets: TensorFlow version 2.16.2 available.
2026-05-11 19:58:35,966 [INFO] sentence_transformers.SentenceTransformer: Load pretrained SentenceTransformer: intfloat/multilingual-e5-large


LLM: cheap=deepseek-v32/latest, strong=deepseek-v32/latest, extraction=deepseek-v32/latest, concurrency=4


2026-05-11 19:58:40,367 [INFO] hierarchical_llm_version.models.embedder: Embedder using input prefix 'query: ' (required by intfloat/multilingual-e5-large family)


Embedder: intfloat/multilingual-e5-large (dim=1024)


## [0] Load text + preprocess + chunk

In [5]:
from hierarchical_llm_version.utils.io import load_text
from hierarchical_llm_version.stages.preprocessing import preprocess
from hierarchical_llm_version.stages.chunking import build_chunks

text = load_text(config.paths.input_text)
print(f'Text: {len(text)} chars')
print(text[:300], '...')

sentences = preprocess(text, language=config.preprocessing.language)
chunks = build_chunks(sentences, config.preprocessing)
print(f'\n{len(sentences)} sentences -> {len(chunks)} chunks')
for c in chunks[:3]:
    print(f'  {c.id} (sents {c.sentence_ids}): {c.text[:120]}...')

Text: 15466 chars
# Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда.

Пусть теперь наши таргеты $y$ кодируют принадлежность к положительн ...

112 sentences -> 28 chunks
  chunk_0 (sents [0, 1, 2, 3, 4]): # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную клас...
  chunk_1 (sents [4, 5, 6, 7, 8]): В этом параграфе договоримся именно так обозначать классы, хотя в жизни вам будут нередко встречаться и метки $\{0,1\}$....
  chunk_2 (sents [8, 9, 10, 11, 12]): Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель классификации, нам ещё предстоит понят...


## [1.1] Pass 1 — chunk summaries (level-1 leaves)

In [6]:
from hierarchical_llm_version.stages.pass1_summarize import summarize_chunks

leaf_nodes = await summarize_chunks(chunks, llm, config.pass1, base_dir=Path('.'))
print(f'Leaves: {len(leaf_nodes)}')
for n in leaf_nodes[:3]:
    kc = ', '.join(f'{c.name}({c.importance})' for c in n.key_concepts[:5])
    print(f'\n{n.id}  topic={n.topic!r}\n  summary: {n.summary[:200]}\n  concepts: {kc}')

2026-05-11 19:59:00,441 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 19:59:15,724 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 19:59:24,145 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 19:59:24,151 [WARNING] hierarchical_llm_version.stages.pass1_summarize: [1.1] chunk_2: parse-empty (summary missing). truncated=False raw_len=0 head=''
2026-05-11 19:59:24,285 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 19:59:24,288 [WARNING] hierarchical_llm_version.stages.pass1_summarize: [1.1] chunk_0: parse-empty (summary missing). truncated=False raw_len=0 head=''
2026-05-11 19:59:24,311 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 19:59:24,315 [INFO] hierarchical_llm_version.stages.pass1_summa

Leaves: 28

h1_0  topic=''
  summary: # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ клас
  concepts: 

h1_1  topic='линейная классификация'
  summary: В этом фрагменте объясняется цель обучения линейной модели для бинарной классификации, где плоскость используется для разделения двух классов. Вводится понятие линейно разделимой выборки, при которой 
  concepts: 

h1_2  topic=''
  summary: Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель классификации, нам ещё предстоит понять, но уже ясно, что итоговое предсказание можно будет вычислить по формуле:

$$

  concepts: 


## [1.2-1.5] Pass 1 — build hierarchy

In [7]:
from hierarchical_llm_version.stages.pass1_hierarchy import build_hierarchy

tree = await build_hierarchy(
    leaf_nodes, llm, config.pass1,
    embedder=embedder if config.pass1.grouping.method == 'semantic' else None,
    base_dir=Path('.'),
)
print(f'Tree: {len(tree.nodes)} nodes, root={tree.root_id}')
for level in sorted(tree.levels):
    print(f'  level {level}: {len(tree.levels[level])} nodes')

root = tree.get(tree.root_id)
print(f'\n=== ROOT ===')
print(f'topic: {root.topic}')
print(f'summary: {root.summary}')
print(f'subtopics: {root.subtopics}')
print(f'core concepts: {[c.name for c in root.key_concepts if c.importance == "core"][:10]}')

2026-05-11 20:02:53,701 [INFO] hierarchical_llm_version.stages.pass1_hierarchy: [1.4] level 2: 28 nodes -> 6 groups


2026-05-11 20:03:23,486 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:03:40,383 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:03:57,004 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:03:57,009 [WARNING] hierarchical_llm_version.stages.pass1_aggregate: [pass1_aggregate_l2] h2_0: parse-empty (summary missing). truncated=False raw_len=0 head=''
2026-05-11 20:04:03,728 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:04:03,732 [WARNING] hierarchical_llm_version.stages.pass1_aggregate: [pass1_aggregate_l2] h2_3: parse-empty (summary missing). truncated=False raw_len=0 head=''
2026-05-11 20:04:26,469 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:04:49,766 [INFO] httpx: HTTP Request

Tree: 35 nodes, root=root
  level 1: 28 nodes
  level 2: 6 nodes
  level 3: 1 nodes

=== ROOT ===
topic: 
summary: # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть те В этом фрагменте объясняется цель обучения линейной модели для бинарной классификации, где плоскость используется для разделения двух классов. Вводится понятие линейно разделимой выборки, при которой плоскость идеально разделяет классы, но отмечается, что в реальности такое встречается редко. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель классификации, нам ещё предстоит понять, но уже ясно, что итоговое предсказание можно будет вычислить по формуле:

$$
y = \operatorname{sign}\langle w, x_i \r Автор указывает на ошибку предсказания, где важен знак числа, а не его модуль, и предлагает сконструировать ф

### Пример собранного контекста для одного chunk

Контекст = root summary + промежуточные узлы пути от root к листу + дедуплицированные ключевые концепты (top-K по `pass2.max_concepts_in_context`) + соседние chunks.

In [8]:
from hierarchical_llm_version.stages.pass2_context import build_context

chunk_to_leaf = {n.chunk_id: n.id for n in tree.nodes.values() if n.level == 1 and n.chunk_id}
chunk_index = {c.id: i for i, c in enumerate(chunks)}

sample = chunks[len(chunks) // 2]
ctx = build_context(sample, chunk_to_leaf, tree, chunks, chunk_index, config.pass2)
print(f'CHUNK {sample.id}:\n{sample.text[:200]}...\n')
print(f'CONTEXT ({len(ctx)} chars):\n{ctx[:1500]}')

CHUNK chunk_14:
Почему же SVM был столь популярен? Из-за небольшого количества параметров и доказуемой оптимальности. Сейчас для нас нормально выбирать специальный алгоритм под задачу и подбирать оптимальные гиперпар...

CONTEXT (3366 chars):
=== Глобальный обзор корпуса (уровень 3) ===
Summary: # Линейная классификация

Теперь давайте поговорим про задачу классификации. Для начала будем говорить про бинарную классификацию на два класса. Обобщить эту задачу до задачи классификации на $K$ классов не составит большого труда. Пусть те В этом фрагменте объясняется цель обучения линейной модели для бинарной классификации, где плоскость используется для разделения двух классов. Вводится понятие линейно разделимой выборки, при которой плоскость идеально разделяет классы, но отмечается, что в реальности такое встречается редко. Увы, в реальной жизни такое встречается крайне редко. Как обучить линейную модель классификации, нам ещё предстоит понять, но уже ясно, что итоговое предсказание можно 

## [2.1-2.2] Pass 2 — extraction with global context

Промпт сейчас в **recall-режиме**: извлекаем все сущности и факты, каждому LLM присваивает numeric `importance ∈ [0,1]`. Фильтрация — отдельным шагом ниже.

In [9]:
from hierarchical_llm_version.stages.pass2_extraction import extract_with_context

entities, triplets = await extract_with_context(
    chunks, tree, llm, config.pass2, base_dir=Path('.'),
)
print(f'Extracted: {len(entities)} entities, {len(triplets)} triplets\n')
print('Sample entities (importance is float 0..1):')
for e in sorted(entities, key=lambda x: -x.importance)[:8]:
    print(f'  {e.importance:.2f}  {e.canonical_name}  [{e.entity_type}]  mention="{e.mention_text}"')
print('\nSample triplets:')
for t in sorted(triplets, key=lambda x: -x.importance)[:8]:
    print(f'  imp={t.importance:.2f} conf={t.confidence:.2f}  {t.subject} | {t.predicate} | {t.object}')
    if t.evidence:
        print(f'      evidence: "{t.evidence[:120]}"')

2026-05-11 20:07:29,644 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:07:29,649 [WARNING] hierarchical_llm_version.stages.pass2_extraction: [2.2] chunk_0: e=0 t=0 truncated=False raw_len=0
  HEAD: ''
  TAIL: ''
2026-05-11 20:07:31,707 [INFO] httpx: HTTP Request: POST https://ai.api.cloud.yandex.net/v1/responses "HTTP/1.1 200 OK"
2026-05-11 20:07:31,878 [WARNING] hierarchical_llm_version.stages.pass2_extraction: [2.2] chunk_2: e=0 t=2 truncated=True raw_len=780
  HEAD: '{   "triplets": [     {       "subject": "линейная модель классификации",       "predicate": "обучается для",       "object": "классификации",       "importance": 0.85,       "confidence": 0.95,       "evidence": "Как обучить линейную модель классификации"     },     {       "subject": "итоговое предсказание",       "predicate": "вычисляется по формуле",       "object": "$y = \\\\operatorname{sign}\\'
  TAIL: '": "задача классификации",       "predicate": "м

Extracted: 22 entities, 40 triplets

Sample entities (importance is float 0..1):
  0.95  $\nabla_w L(y,X,w)= -\sum_i x_i\left(y_i-\sigma(\langle w,x_i\rangle)\right)$  [формула]  mention="$\nabla_w L(y,X,w)= -\sum_i x_i\left(y_i-\sigma(\langle w,x_i\rangle)\right)$"
  0.90  логистическая регрессия  [метод]  mention="логистическая регрессия"
  0.90  $p=\sigma(\langle w,x_i\rangle)$  [формула]  mention="$p=\sigma(\langle w,x_i\rangle)$"
  0.85  сигмоида  [функция]  mention="сигмоида"
  0.85  $\frac{d\log\sigma(z)}{dz}= \sigma(-z)$  [формула]  mention="$\frac{d\log\sigma(z)}{dz}= \sigma(-z)$"
  0.85  $\frac{d\log\sigma(-z)}{dz}=-\sigma(z)$  [формула]  mention="$\frac{d\log\sigma(-z)}{dz}=-\sigma(z)$"
  0.80  градиентный спуск  [метод]  mention="градиентный спуск"
  0.80  $\nabla_w \log\sigma(\langle w,x_i\rangle)=\sigma(-\langle w,x_i\rangle)x_i$  [формула]  mention="$\nabla_w \log\sigma(\langle w,x_i\rangle)=\sigma(-\langle w,x_i\rangle)x_i$"

Sample triplets:
  imp=0.90 conf=0.95  итого

## [2.3] Entity resolution

In [10]:
from hierarchical_llm_version.stages.entity_resolution import resolve_entities

mapping, entities, triplets = resolve_entities(
    entities, triplets, embedder, config.entity_resolution,
)
print(f'Rewrites: {len(mapping)}')
for orig, resolved in list(mapping.items())[:10]:
    print(f'  {orig!r:40} -> {resolved!r}')

2026-05-11 20:16:57,171 [INFO] hierarchical_llm_version.stages.entity_resolution: [2.3] ER cluster sizes (top): [39, 19, 3, 2, 2, 1, 1, 1, 1, 1]
2026-05-11 20:16:57,181 [WARNING] hierarchical_llm_version.stages.entity_resolution: [2.3] Suspicious ER collapse: largest cluster has 39/71 names. Sample: ['логистическая регрессия', 'линейная регрессия', 'явная формула решения', 'градиентный спуск', 'градиент', 'вывод формулы градиента', 'предсказание модели', 'функция потерь']. Likely causes: (a) embedding model needs an input prefix (e5/bge -> 'query: '); (b) similarity_threshold (0.85) too low; (c) transitive union-find chain.
2026-05-11 20:16:57,186 [INFO] hierarchical_llm_version.stages.entity_resolution: [2.3] Entity resolution: 71 unique names -> 11 after merge (60 rewrites)


Rewrites: 60
  'логистическая регрессия'                -> 'сумма'
  'линейная регрессия'                     -> 'сумма'
  'явная формула решения'                  -> 'сумма'
  'градиентный спуск'                      -> 'сумма'
  'градиент'                               -> 'сумма'
  '$\\nabla_w L(y,X,w)= -\\sum_i x_i\\left(y_i-\\sigma(\\langle w,x_i\\rangle)\\right)$' -> '$p$'
  'вывод формулы градиента'                -> 'сумма'
  '$\\frac{d\\log\\sigma(z)}{dz}= \\sigma(-z)$' -> '$p$'
  '$\\frac{d\\log\\sigma(-z)}{dz}=-\\sigma(z)$' -> '$p$'
  '$\\nabla_w \\log\\sigma(\\langle w,x_i\\rangle)=\\sigma(-\\langle w,x_i\\rangle)x_i$' -> '$p$'


## [4] Raw graph (с importance на узлах и рёбрах)

Узел.importance = max по entities этого canonical_name + max по incident triplets.
Ребро.importance = max по triplets этого ключа.

In [11]:
from hierarchical_llm_version.stages.graph_assembly import assemble_raw_graph

raw_graph = assemble_raw_graph(entities, triplets, chunks, text, config)
print(f'Raw graph (full, no filter): {len(raw_graph.nodes)} nodes, {len(raw_graph.edges)} edges')
print('\nTop nodes by importance:')
for n in sorted(raw_graph.nodes, key=lambda x: -x.importance)[:8]:
    print(f'  imp={n.importance:.2f}  {n.label}  ({len(n.mentions)} mentions)')
print('\nTop edges by importance:')
for e in sorted(raw_graph.edges, key=lambda x: -x.importance)[:8]:
    print(f'  imp={e.importance:.2f}  {e.source} --[{e.label}]--> {e.target}  (w={e.weight})')

Raw graph (full, no filter): 11 nodes, 34 edges

Top nodes by importance:
  imp=0.95  $p$  (19 mentions)
  imp=0.90  сумма  (66 mentions)
  imp=0.85  сигмоида  (5 mentions)
  imp=0.50  скалярное произведение  (1 mentions)
  imp=0.50  отступ  (2 mentions)
  imp=0.50  кусочно-постоянная функция  (1 mentions)
  imp=0.50  ноль  (1 mentions)
  imp=0.50  мажорирование  (1 mentions)

Top edges by importance:
  imp=0.50  n0 --[обучается для]--> n0  (w=1)
  imp=0.50  n0 --[вычисляется по формуле]--> n1  (w=1)
  imp=0.50  n5 --[вычисляет]--> n0  (w=1)
  imp=0.50  n0 --[является]--> n6  (w=1)
  imp=0.50  n0 --[невозможно оптимизировать]--> n0  (w=1)
  imp=0.50  n0 --[равна]--> n7  (w=1)
  imp=0.50  n0 --[мажорируется]--> n0  (w=1)
  imp=0.50  n0 --[можно решить]--> n8  (w=1)


## [5] Importance filtering — single + multi (sweep)

Single режим — один порог `(t_e, t_r)` из `default_*_threshold`.
Multi режим — сетка `(t_e, t_r)`; каждый вариант → отдельный `ClusteredGraph`. Получаем `MultiClusteredGraph` в формате `llm_v2`, под который рендерится `viewer_cytoscape.py` со слайдером.

In [12]:
from hierarchical_llm_version.stages.importance_filter import (
    build_clustered_from_filter,
    build_multi_clustered,
)

ifc = config.importance_filtering
multi_clustered = None
default_label = None

if ifc.enabled and ifc.multi.enabled:
    multi_clustered, default_label = build_multi_clustered(raw_graph, ifc.multi)
    method = multi_clustered.methods['importance_filter']
    print(f'Multi: {len(method.param_labels)} variants. Default = {default_label}\n')
    for lbl in method.param_labels:
        g = method.graphs[lbl]
        print(f'  {lbl:20}  nodes={len(g.nodes):4}  edges={len(g.edges):4}')
    clustered = method.graphs[default_label]
elif ifc.enabled:
    clustered = build_clustered_from_filter(
        raw_graph,
        entity_threshold=ifc.default_entity_threshold,
        relation_threshold=ifc.default_relation_threshold,
        drop_orphan_nodes=ifc.drop_orphan_nodes,
    )
    print(f'Single (e>={ifc.default_entity_threshold}, r>={ifc.default_relation_threshold}): {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')
else:
    from hierarchical_llm_version.stages.build_clustered import build_clustered_graph
    clustered = build_clustered_graph(raw_graph, config.clustering, embedder=embedder)
    print(f'No importance filter. Clustered: {len(clustered.nodes)} nodes, {len(clustered.edges)} edges')

print('\nClustered top-5 nodes:')
for n in clustered.nodes[:5]:
    print(f'  imp={n.importance:.2f}  {n.id}: {n.label}')

Multi: 16 variants. Default = e=0.60,r=0.20

  e=0.20,r=0.20         nodes=   9  edges=  34
  e=0.20,r=0.40         nodes=   9  edges=  34
  e=0.20,r=0.60         nodes=   0  edges=   0
  e=0.20,r=0.80         nodes=   0  edges=   0
  e=0.40,r=0.20         nodes=   9  edges=  34
  e=0.40,r=0.40         nodes=   9  edges=  34
  e=0.40,r=0.60         nodes=   0  edges=   0
  e=0.40,r=0.80         nodes=   0  edges=   0
  e=0.60,r=0.20         nodes=   3  edges=  24
  e=0.60,r=0.40         nodes=   3  edges=  24
  e=0.60,r=0.60         nodes=   0  edges=   0
  e=0.60,r=0.80         nodes=   0  edges=   0
  e=0.80,r=0.20         nodes=   3  edges=  24
  e=0.80,r=0.40         nodes=   3  edges=  24
  e=0.80,r=0.60         nodes=   0  edges=   0
  e=0.80,r=0.80         nodes=   0  edges=   0

Clustered top-5 nodes:
  imp=0.90  c0: сумма
  imp=0.95  c1: $p$
  imp=0.85  c2: сигмоида


## Save outputs + статистика

In [13]:
from hierarchical_llm_version.utils.io import save_json

out = Path(config.paths.output_dir)
out.mkdir(parents=True, exist_ok=True)

save_json(tree.model_dump(), out / 'hierarchy_tree.json')
save_json({'entities': [e.model_dump() for e in entities],
           'triplets': [t.model_dump() for t in triplets],
           'mapping': mapping}, out / 'extracted_resolved.json')
save_json(raw_graph.model_dump(), out / 'raw_graph.json')
save_json(clustered.model_dump(), out / 'clustered_graph.json')
if multi_clustered is not None:
    save_json(multi_clustered.model_dump(), out / 'multi_clustered_graph.json')
save_json(llm.stats.to_dict(), out / 'token_stats.json')

stats = llm.stats.to_dict()
print(f"Tokens — input: {stats['total_input_tokens']}, output: {stats['total_output_tokens']}, calls: {stats['total_calls']}, failures: {stats['total_failures']}")
print('\nBy stage:')
for stage, s in stats['by_stage'].items():
    print(f"  {stage:30}  in={s['input_tokens']:7}  out={s['output_tokens']:7}  calls={s['n_calls']}")

print('\nHierarchy levels:')
for lvl in sorted(tree.levels):
    print(f'  level {lvl}: {len(tree.levels[lvl])} nodes')

if multi_clustered is not None:
    print('\nMulti-clustered variants saved (slider-ready in llm_v2 viewer):')
    method = multi_clustered.methods['importance_filter']
    for lbl in method.param_labels:
        g = method.graphs[lbl]
        print(f'  {lbl:20}  nodes={len(g.nodes):4}  edges={len(g.edges):4}')

print(f'\nSaved to {out.resolve()}/')

Tokens — input: 106108, output: 169718, calls: 63, failures: 0

By stage:
  pass1_summarize                 in=  13689  out=  39271  calls=28
  pass1_aggregate_l2              in=   4626  out=  15447  calls=6
  pass1_root                      in=   1738  out=   3000  calls=1
  pass2_extraction                in=  86055  out= 112000  calls=28

Hierarchy levels:
  level 1: 28 nodes
  level 2: 6 nodes
  level 3: 1 nodes

Multi-clustered variants saved (slider-ready in llm_v2 viewer):
  e=0.20,r=0.20         nodes=   9  edges=  34
  e=0.20,r=0.40         nodes=   9  edges=  34
  e=0.20,r=0.60         nodes=   0  edges=   0
  e=0.20,r=0.80         nodes=   0  edges=   0
  e=0.40,r=0.20         nodes=   9  edges=  34
  e=0.40,r=0.40         nodes=   9  edges=  34
  e=0.40,r=0.60         nodes=   0  edges=   0
  e=0.40,r=0.80         nodes=   0  edges=   0
  e=0.60,r=0.20         nodes=   3  edges=  24
  e=0.60,r=0.40         nodes=   3  edges=  24
  e=0.60,r=0.60         nodes=   0  edges=  